# 07 — Aggregated Results & Error Analysis

Combines test predictions from every model into one comparison frame, then drills into:
1. Side-by-side metrics table for the report
2. Confusion matrices grid
3. ROC curves
4. Error breakdown by tweet features (length, hashtags, mentions, emojis) — folds in the lab's emoji/engagement sub-questions as **explanatory variables** for misclassification
5. Qualitative example surface (10 hardest tweets per model)

**Run after** notebooks 01–04 locally and 05–06 on Colab (with `predictions_distilbert.parquet` and `predictions_climatebert.parquet` downloaded into `results/predictions/`).

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve

from src.data import load_splits
from src.eval import LABELS

FIG_DIR = PROJECT_ROOT / 'results' / 'figures' / 'analysis'
FIG_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR = PROJECT_ROOT / 'results' / 'predictions'

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

## 1. Load metrics and predictions

In [ ]:
MODEL_ORDER = [
    'majority', 'tfidf_lr', 'bilstm_glove', 'lab_transformer', 'distilbert', 'climatebert',
]
PRETTY = {
    'majority': 'Majority',
    'tfidf_lr': 'TF-IDF + LR',
    'bilstm_glove': 'BiLSTM+GloVe',
    'lab_transformer': 'Lab Transformer',
    'distilbert': 'DistilBERT (FT)',
    'climatebert': 'ClimateBERT (FT)',
}

metrics = pd.read_csv(PROJECT_ROOT / 'results' / 'metrics.csv')
metrics = metrics.set_index('model').reindex([m for m in MODEL_ORDER if m in metrics.index])
metrics.round(4)

In [ ]:
preds = {}
for m in MODEL_ORDER:
    p = PRED_DIR / f'{m}.parquet'
    if p.exists():
        preds[m] = pd.read_parquet(p)
        print(f'{m:<18}  loaded ({len(preds[m]):,} rows)')
    else:
        print(f'{m:<18}  MISSING — run the corresponding notebook first')

## 2. Side-by-side metrics table

In [ ]:
show_cols = ['accuracy', 'f1_macro', 'f1_activist', 'f1_sceptic', 'roc_auc']
summary = metrics[show_cols].copy()
summary.index = [PRETTY.get(m, m) for m in summary.index]
summary = summary.round(4)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
summary[['accuracy', 'f1_macro']].plot(kind='bar', ax=ax, width=0.8)
ax.set_ylim(0.4, 1.0)
ax.set_ylabel('Score')
ax.set_title('Test-set performance across models')
ax.set_xticklabels(summary.index, rotation=20, ha='right')
plt.tight_layout()
plt.savefig(FIG_DIR / 'model_comparison.png', bbox_inches='tight')
plt.show()

## 3. Confusion matrix grid

In [ ]:
available = [m for m in MODEL_ORDER if m in preds]
n = len(available)
cols = 3
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.6 * rows))
axes = np.atleast_2d(axes).ravel()

for i, m in enumerate(available):
    df = preds[m]
    cm = confusion_matrix(df['y_true'], df['y_pred'], labels=LABELS)
    sns.heatmap(
        cm, annot=True, fmt=',d', cmap='Blues',
        xticklabels=LABELS, yticklabels=LABELS, cbar=False, ax=axes[i],
    )
    axes[i].set_title(PRETTY.get(m, m))
    axes[i].set_xlabel('Predicted'); axes[i].set_ylabel('True')

for j in range(n, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.savefig(FIG_DIR / 'confusion_grid.png', bbox_inches='tight')
plt.show()

## 4. ROC curves

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for m in available:
    df = preds[m]
    if 'proba_sceptic' not in df.columns:
        continue
    y_bin = (df['y_true'] == 'sceptic').astype(int)
    fpr, tpr, _ = roc_curve(y_bin, df['proba_sceptic'])
    auc_val = metrics.loc[m, 'roc_auc'] if 'roc_auc' in metrics.columns else None
    label = f'{PRETTY[m]}' + (f' (AUC={auc_val:.3f})' if pd.notna(auc_val) else '')
    ax.plot(fpr, tpr, label=label)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax.set_xlabel('False positive rate')
ax.set_ylabel('True positive rate')
ax.set_title('ROC curves (sceptic = positive)')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig(FIG_DIR / 'roc_curves.png', bbox_inches='tight')
plt.show()

## 5. Error breakdown by tweet features

Folds in the lab's emoji/engagement sub-questions: rather than treat them as separate tasks, we use them as **explanatory variables** for *where* models fail.

In [ ]:
_, _, test_df = load_splits()
test_df = test_df.reset_index(drop=True)
test_df['len_bucket'] = pd.cut(
    test_df['clean_len'], bins=[-1, 5, 15, 30, 1000],
    labels=['<=5', '6-15', '16-30', '31+'],
)
test_df['has_hashtag'] = test_df['n_hashtags'] > 0

# Build a long-format error table: one row per (model, feature_value) with error rate
rows = []
for m in available:
    df = preds[m].reset_index(drop=True)
    if len(df) != len(test_df):
        print(f'skip {m} (len mismatch: {len(df)} vs {len(test_df)})')
        continue
    err = (df['y_true'].values != df['y_pred'].values).astype(int)
    tmp = test_df.assign(err=err, model=PRETTY.get(m, m))
    rows.append(tmp)

if rows:
    long = pd.concat(rows, ignore_index=True)
    print(long.head())

In [ ]:
if rows:
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))

    by_len = long.groupby(['model', 'len_bucket'], observed=True)['err'].mean().reset_index()
    sns.barplot(data=by_len, x='len_bucket', y='err', hue='model', ax=axes[0])
    axes[0].set_title('Error rate by tweet length (words)')
    axes[0].set_ylabel('error rate')
    axes[0].set_xlabel('cleaned length')
    axes[0].legend(fontsize=7, loc='upper right')

    by_hash = long.groupby(['model', 'has_hashtag'], observed=True)['err'].mean().reset_index()
    sns.barplot(data=by_hash, x='has_hashtag', y='err', hue='model', ax=axes[1])
    axes[1].set_title('Error rate by hashtag presence')
    axes[1].set_ylabel('error rate')
    axes[1].set_xlabel('original tweet had a hashtag')
    axes[1].legend(fontsize=7, loc='upper right')

    by_emoji = long.groupby(['model', 'has_emoji'], observed=True)['err'].mean().reset_index()
    sns.barplot(data=by_emoji, x='has_emoji', y='err', hue='model', ax=axes[2])
    axes[2].set_title('Error rate by emoji presence')
    axes[2].set_ylabel('error rate')
    axes[2].set_xlabel('original tweet had emoji')
    axes[2].legend(fontsize=7, loc='upper right')

    plt.tight_layout()
    plt.savefig(FIG_DIR / 'error_by_feature.png', bbox_inches='tight')
    plt.show()

## 6. Hardest examples (most-confident wrongs) per model

In [ ]:
for m in available:
    df = preds[m].reset_index(drop=True)
    if 'proba_sceptic' not in df.columns:
        continue
    if len(df) != len(test_df):
        continue
    df = df.assign(text=test_df['text'].values, clean_text=test_df['clean_text'].values)
    wrong = df[df['y_true'] != df['y_pred']].copy()
    # confidence = distance from 0.5
    wrong['confidence'] = (wrong['proba_sceptic'] - 0.5).abs()
    top = wrong.nlargest(8, 'confidence')[['y_true', 'y_pred', 'proba_sceptic', 'text']]
    print(f'\n=== {PRETTY.get(m, m)} — most-confident misclassifications ===')
    for _, row in top.iterrows():
        snippet = (row['text'] or '')[:140].replace('\n', ' ')
        print(f'  true={row["y_true"]:<8} pred={row["y_pred"]:<8} p_sceptic={row["proba_sceptic"]:.2f}  {snippet}')

## 7. Save aggregated metrics for the report

In [ ]:
summary.to_csv(PROJECT_ROOT / 'results' / 'summary_table.csv')
metrics.round(4).to_csv(PROJECT_ROOT / 'results' / 'full_metrics_table.csv')
print('Saved:')
print(f'  results/summary_table.csv      ({len(summary)} rows)')
print(f'  results/full_metrics_table.csv ({len(metrics)} rows)')